(sec:ensembles)=
# Ensembles

To treat a QM region embedded in a complex environment composed of proteins, water, and ions, the methods described in this section can be useful. Such environments can be represented explicitly using:

- Polarizable embedding (PE), where molecules in the environment are represented by site charges and polarizabilities.

- Non-polarizable embedding (NPE), where the environment is represented by point charges only.


A PDB file, containing one single structure or several structures, can be passed to the `EnsembleParser`. At this stage, the PE and NPE cutoffs (in Å) are selected:

In [ ]:
import veloxchem as vlx

ens_parser = vlx.EnsembleParser()

ensemble = ens_parser.structures(
    trajectory_file = "../input_files/alpha-helix-acetone-water.pdb",
    qm_region = "resname LIG",
    env_region = "protein or water or resname NA CL",
    pe_cutoff = 2.5,
    npe_cutoff = 10.0,
)

Key parameters are the following:

- `qm_region`: MDAnalysis selection string that defines QM region. In the above example, all atoms in residue name `LIG` will be treated as QM. Larger QM regions can be selected, e.g. 

   ```resname LIG or byres (water and around 3.0 (resname LIG))``` 

  will treat as QM all atoms in `LIG` as well as all atoms in water molecules within 3 Å from the center of mass of `LIG`. 


- `qm_charge`: charge of the QM region

- `env_region`: MDAnalysis selection string that defines environment. In the above example, all atoms in the protein, water and Na$^{+}$ and Cl$^{-}$ ions will be considered. In the environment region:

    - All residues within a `pe_cutoff` from the center of mass of the QM region will be treated with PE.

    - All residues within a `npe_cutoff` from the center of mass of the QM region will be treated with NPE.

- If `pe_cutoff = npe_cutoff = 0`, then all the system will be treated as QM.


A time-resolved trajectory can also be provided as follows:

In [ ]:
ensemble = ens_parser.structures(
    trajectory_file = "../input_files/alpha-helix-acetone-water.xtc",
    topology_file = "../input_files/alpha-helix-acetone-water.tpr",
    qm_region = "resname LIG",
    num_snapshots = 3,
    pe_cutoff = 2.5,
    npe_cutoff = 10.0,
)

where the following parameters are useful:

- `num_snapshots`: Controls snapshot extraction.

    - _Default_: All snapshots used.

    - _If provided_: The specified number of snapshots is selected at evenly spaced intervals.

- `start`: Start time of the trajectory window in ps. By default, the time of the first snapshot is used.

- `end`: End time of the trajectory window in ps. By default, the time of the last snapshot is used.

- `last_snapshot_only = True`: Processes only the final snapshot.

The number of residues treated with PE and NPE can be accessed as follows:

In [4]:
print("number residues PE = ", ensemble[0]["number_residues_pe"])
print("number residues NPE = ", ensemble[0]["number_residues_npe"])

number residues PE =  3
number residues NPE =  175


Once the environment has been defined, the environment models can be selected with the `set_env_models` method of the `EnsembleDriver`:

In [ ]:
ens_drv = vlx.EnsembleDriver()

ens_drv.set_env_models(
    pe_model=["CP3", "SEP"],
    npe_model=["ff19sb", "tip3p"],
)

VeloxChem supports the following environment models for PE and NPE:
- PE: `CP3` {cite}`Reinholdt2020` for proteins, and `SEP` {cite}`Beerepoot2016` for common polar and non-polar solvent molecules and ions.
- NPE: `ff19sb` {cite}`Tian2020` for proteins, and `tip3p` {cite}`Jorgensen1983` for water.


:::{image} ../images/models-no-cite.png
:align: center
:width: 600px
:::

In some situations, for example when preparing input files for running on a cluster, it is convenient to use the `write_pot_files` method to generate the potential files:

In [ ]:
ens_drv.write_pot_files(ensemble)

The SCF calculations can be automated by passing `scf_options` and, optionally, `property_options` to the `compute` method:

In [ ]:
scf_options = {
   "scf_type": "restricted",
   "conv_thresh": 1.0e-6,
   "max_iter": 150,
   "xcfun": "cam-b3lyp",
   "grid_level": 4,
}

property_options = {
    "property": "absorption",
    "nstates": 5,
    "nto": True,
}

In [ ]:
results = ens_drv.compute(
   ensemble,
   basis_set = "def2-svp",
   scf_options = scf_options,
   property_options = property_options,
)

Finally, the averaged spectra can be plotted:

In [ ]:
ens_drv.plot_uv_vis_spectra(
    results,
    show_individual = True,
    show_sticks = True,
    xlim_nm = (150, 275)
)

:::{image} ../images/averaged_spectra_1.png
:align: center
:width: 600px
:::

**Text file**

:::{code}
@jobs
task: scf
@end

@method settings
xcfun: cam-b3lyp
basis: def2-svp
potfile: pe_frame_000000.pot ! environment NPE/PE charges & polarizabilities.
@end

@molecule
charge: 0
multiplicity: 1
xyz:  ! qm_coordinates:
H  23.09239578  21.92239571  22.43439484
C  23.3823967   22.95239449  22.65439606
H  24.33239555  23.14239693  22.14439583
H  22.6023941   23.562397    22.18439484
C  23.44239616  23.26239395  24.09439468
O  23.01239586  22.44239616  24.92439651
C  24.08239555  24.58239555  24.50439644
H  24.36239624  24.5623951   25.55439568
H  24.92239571  24.80239487  23.84439659
H  23.46239662  25.46239471  24.35439491
@end
:::


The potential file named `pe_frame_000000.pot` in this example takes the following form: [`pe_frame_000000.pot`](../input_files/pe_frame_000000.pot)